#Youtube comments analysis

##Introduction <br>
The purpose of this project is to analyze YouTube comments on chosen topic. It explores the presence of positive, neutral andnegative sentiment and how certain topics and words correlate with user reactions.

Installing required packages

In [ ]:
pip install --upgrade google-api-python-client

In [ ]:
pip install --upgrade google-auth-oauthlib google-auth-httplib2

In [ ]:
pip install numpy pandas scikit-learn gensim matplotlib seaborn plotly sentence-transformers bertopic

In [ ]:
pip install pyLDAvis transformers umap-learn hdbscan tomotopy spacy langdetect  datasets torch

##Analysis under chosen video

In [ ]:
import googleapiclient.discovery
import pandas as pd

api_service_name = "youtube"
api_version = "v3"
DEVELOPER_KEY = "###your developer key there###"

youtube = googleapiclient.discovery.build(
    api_service_name, api_version, developerKey=DEVELOPER_KEY)

request = youtube.commentThreads().list(
    part="snippet",
    videoId="###video ID chosen by you###",
    maxResults=100
)
response = request.execute()

comments = []

for item in response['items']:
    comment = item['snippet']['topLevelComment']['snippet']
    comments.append([
        comment['authorDisplayName'],
        comment['publishedAt'],
        comment['updatedAt'],
        comment['likeCount'],
        comment['textDisplay']
    ])

df = pd.DataFrame(comments, columns=['author', 'published_at', 'updated_at', 'like_count', 'text'])

df.head(10)

Here I did analysis of comments from two weeks of uploading the video for relevance, you can change this, or simply not do it.

In [ ]:
import pandas as pd
from googleapiclient.discovery import build
from datetime import datetime, timedelta

def get_video_comments(video_id, published_date):
    comments = []
    try:
        two_weeks_later = datetime.strptime(published_date, "%Y-%m-%dT%H:%M:%S%z") + timedelta(weeks=2)

        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            textFormat="plainText",
            maxResults=100
        )

        while request:
            response = request.execute()

            if 'items' not in response:
                print(f"No comments found for video {video_id}.")
                break

            for item in response['items']:
                comment = item['snippet']['topLevelComment']['snippet']
                comment_text = comment['textDisplay']
                comment_date = comment['publishedAt']

                #comments within the 2-week period
                comment_datetime = datetime.strptime(comment_date, "%Y-%m-%dT%H:%M:%S%z")
                if comment_datetime <= two_weeks_later:
                    comments.append({
                        "video_id": video_id,
                        "comment_text": comment_text,
                        "comment_date": comment_date
                    })

            #next page comments if there are any
            if 'nextPageToken' in response:
                request = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    textFormat="plainText",
                    maxResults=100,
                    pageToken=response['nextPageToken']
                )
            else:
                break
    except Exception as e:
        print(f"Error retrieving comments for video {video_id}: {e}")

    return comments
###your video IDs
video_ids = ["###video 1 ID###", "###video 2 ID###"]
all_comments = []

for video_id in video_ids:
    print(f"Processing video: {video_id}")
    published_date = get_video_details(video_id)

    if published_date:
        comments = get_video_comments(video_id, published_date)
        all_comments.extend(comments)
    else:
        print(f"Video {video_id} not found or published date missing.")

df_comments = pd.DataFrame(all_comments)
df_comments.to_csv('youtube_comments.csv', index=False)
print(df_comments.head())

Video ID name change - you can do it for clearer information

In [ ]:
video_id_replacements = {
    '###video 1 ID### ': '###name which you would like to use###',
    '###video 2 ID### ': '###name which you would like to use###'
}

It you want to add a video with a different date, you can do it like this

In [ ]:
youtube = build('youtube', 'v3', developerKey='###your developer key###')
def get_video_details(video_id):
    try:
        request = youtube.videos().list(
            part="snippet",
            id=video_id
        )
        response = request.execute()

        if 'items' in response and len(response['items']) > 0:
            video_details = response['items'][0]['snippet']
            return video_details.get('publishedAt')
        else:
            print(f"Error: Video {video_id} not found.")
            return None
    except Exception as e:
        print(f"Error retrieving video details for {video_id}: {e}")
        return None

def get_video_comments(video_id, published_date):
    comments = []
    try:
        published_datetime = datetime.strptime(published_date, "%Y-%m-%dT%H:%M:%S%z")
        two_weeks_later = published_datetime + timedelta(weeks=2)

        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            textFormat="plainText",
            maxResults=100
        )

        while request:
            response = request.execute()

            if 'items' not in response:
                print(f"No comments found for video {video_id}.")
                break

            for item in response['items']:
                comment = item['snippet']['topLevelComment']['snippet']
                comment_text = comment['textDisplay']
                comment_date = comment['publishedAt']

                comment_datetime = datetime.strptime(comment_date, "%Y-%m-%dT%H:%M:%S%z")
                if published_datetime <= comment_datetime <= two_weeks_later:
                    comments.append({
                        "video_id": video_id,
                        "comment_text": comment_text,
                        "comment_date": comment_date
                    })

            if 'nextPageToken' in response:
                request = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    textFormat="plainText",
                    maxResults=100,
                    pageToken=response['nextPageToken']
                )
            else:
                break
    except Exception as e:
        print(f"Error retrieving comments for video {video_id}: {e}")

    return comments

def add_comments_to_existing_db(video_id, start_date, existing_db_path='youtube_comments.csv'):
    new_comments = get_video_comments(video_id, start_date)

    if new_comments:
        df_existing = pd.read_csv(existing_db_path)

        df_new_comments = pd.DataFrame(new_comments)

        df_updated = pd.concat([df_existing, df_new_comments], ignore_index=True)

        df_updated['video_id'] = df_updated['video_id'].replace(video_id_replacements)

        df_updated.to_csv(existing_db_path, index=False)

        print(f"Added {len(new_comments)} new comments to the database and updated video IDs.")
    else:
        print(f"No new comments found for video {video_id} within the specified time period.")

video_id = "###video 3 ID###"
start_date = "2023-10-03T14:45:26Z" ###choose any date you like
add_comments_to_existing_db(video_id, start_date)

Now it's time to remore stop words, you can use a text file list.

In [ ]:
import pandas as pd

with open('###yourtextfile###.txt', 'r', encoding='utf-8') as file:
    polish_stopwords = set(file.read().splitlines())
comments_column = 'comment_text'
text_data = pd.read_csv('youtube_comments.csv')

def remove_stopwords(text, stopwords):
    words = text.split()
    filtered_words = [word for word in words if word not in stopwords]
    return " ".join(filtered_words)

text_data[comments_column] = text_data[comments_column].apply(lambda x: remove_stopwords(str(x), polish_stopwords))
text_data.to_csv('youtube_comments_no_stopwords.csv', index=False)

In [ ]:
text_data = text_data[text_data[comments_column].str.strip().astype(bool)]

##Stemming

In [ ]:
import re
from nltk.stem import PorterStemmer
with open('###yourtextfile###.txt.txt', 'r', encoding='utf-8') as file:
    polish_stopwords = set(file.read().splitlines())

comments_column = 'comment_text'
text_data = pd.read_csv('youtube_comments.csv')

ps = PorterStemmer()
def custom_tokenizer(text):
    return re.findall(r'\b\w+\b', text)

def preprocess_text(text):
    text = str(text)
    tokens = custom_tokenizer(text)
    stemmed_tokens = [ps.stem(word) for word in tokens if word.lower() not in polish_stopwords]
    return " ".join(stemmed_tokens)

text_data[comments_column] = text_data[comments_column].apply(lambda x: preprocess_text(x))

##Word frequency graph

In [ ]:
import re
import pandas as pd
import unicodedata
from nltk.tokenize import word_tokenize
from collections import Counter
def term_frequency(text):
    wordlist = text.split()
    wordfreq = Counter(wordlist)
    return pd.DataFrame(list(wordfreq.items()), columns=["Word", "Frequency"]).sort_values(by="Frequency", ascending=False).reset_index(drop=True)

all_comments = " ".join(text_data[comments_column])
tf_matrix = term_frequency(all_comments)

text_data.to_csv("youtube_comments_cleaned_stemmed_no_stopwords.csv", index=False)
tf_matrix.to_csv("term_frequency_matrix_no_stopwords.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 6))
sns.barplot(x=tf_matrix["Frequency"].head(20), y=tf_matrix["Word"].head(20), palette="viridis")
plt.title("Top 20 Most Frequent Words")
plt.xlabel("Frequency")
plt.ylabel("Words")
plt.show()

From term frequency matrix you can find some words worth adding to stop words.

In [ ]:
from wordcloud import WordCloud
wordcloud = WordCloud(
    width=800, height=400, background_color="white", colormap="viridis"
).generate_from_frequencies(dict(zip(tf_matrix["Word"], tf_matrix["Frequency"])))
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Term Frequencies")
plt.show()

Add some filtering if you need it.

In [ ]:
additional_excluded_words = ['word1','word2','word3']  #my words
excluded_words = tf_matrix["Word"].head(15).tolist() + additional_excluded_words
tf_matrix_filtered = tf_matrix[~tf_matrix["Word"].isin(excluded_words)].reset_index(drop=True)
tf_matrix_filtered.to_csv("tf_matrix_filtered.csv", index=False)

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
word_freq_dict_filtered = dict(zip(tf_matrix_filtered["Word"], tf_matrix_filtered["Frequency"]))
wordcloud = WordCloud(width=800, height=400, background_color="white", colormap="viridis").generate_from_frequencies(word_freq_dict_filtered)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud Excluding Specific Words", fontsize=16)
plt.show()

##LSA

In [ ]:
comments_column = 'comment_text'
text_data = pd.read_csv('youtube_comments_cleaned_stemmed_no_stopwords.csv')

In [ ]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')
text_data["tokenized_comments"] = text_data[comments_column].apply(word_tokenize)

In [ ]:
text_data["filtered_comments"] = text_data["tokenized_comments"].apply(
    lambda x: [word for word in x if word not in polish_stopwords])

In [ ]:
text_data["processed_comments"] = text_data["filtered_comments"].apply(" ".join)

Optimal topic number

In [ ]:
from gensim.corpora.dictionary import Dictionary
from gensim.models import LdaModel, CoherenceModel
from sklearn.feature_extraction.text import CountVectorizer
from tqdm import tqdm


def find_optimal_topics(processed_comments, min_topics=2, max_topics=15, step=1):
    tokenized_data = [doc.split() for doc in processed_comments]
    dictionary = Dictionary(tokenized_data)
    corpus = [dictionary.doc2bow(text) for text in tokenized_data]
    coherence_scores = {}
    for num_topics in tqdm(range(min_topics, max_topics + 1, step), desc="Finding optimal topics"):
        lda_model = LdaModel(corpus=corpus, num_topics=num_topics, id2word=dictionary, random_state=42)

        coherence_model = CoherenceModel(model=lda_model, texts=tokenized_data, dictionary=dictionary, coherence='c_v')
        coherence_score = coherence_model.get_coherence()
        coherence_scores[num_topics] = coherence_score

    return coherence_scores

coherence_results = find_optimal_topics(text_data["processed_comments"])
optimal_topics = max(coherence_results, key=coherence_results.get)
print(f"Optimal number of topics: {optimal_topics}")

In [ ]:
import matplotlib.pyplot as plt

def plot_coherence_scores(coherence_scores):
    plt.figure(figsize=(10, 6))
    plt.plot(list(coherence_scores.keys()), list(coherence_scores.values()), marker='o')
    plt.title("Coherence Score by Number of Topics")
    plt.xlabel("Number of Topics")
    plt.ylabel("Coherence Score")
    plt.grid()
    plt.show()
plot_coherence_scores(coherence_results)

Looking at the topics

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(text_data["processed_comments"])
n_topics = 3
lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
lsa_matrix = lsa_model.fit_transform(tfidf_matrix)

terms = tfidf_vectorizer.get_feature_names_out()
for i, comp in enumerate(lsa_model.components_):
    top_terms = [terms[idx] for idx in comp.argsort()[-10:]]
    print(f"Topic {i + 1}: {', '.join(top_terms)}")

Let's clean the text.

In [ ]:
import re
import unicodedata
from nltk.stem import PorterStemmer
import pandas as pd
comments_column = 'comment_text'
text_data = pd.read_csv('youtube_comments_cleaned_stemmed_no_stopwords.csv')
ps = PorterStemmer()

def remove_diacritics(text):
    if not isinstance(text, str):
        return ""
    return ''.join(
        char for char in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(char)
    )

def custom_tokenizer(text):
    return re.findall(r'\b\w+\b', text)

def clean_and_stem_text(text):
    if not isinstance(text, str):
        return ""

    text = remove_diacritics(text)

    text = re.sub(r'[^a-zA-Z0-9 \n\.]', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(' +', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = text.lower()
    tokens = custom_tokenizer(text)
    stemmed_tokens = [ps.stem(word) for word in tokens if word.lower() not in polish_stopwords]
    return " ".join(stemmed_tokens)
text_data[comments_column] = text_data[comments_column].apply(clean_and_stem_text)
text_data.to_csv('youtube_comments_clean_stemmed.csv', index=False)

Tokenization - we will use it later

In [ ]:
text_data["tokenized_comments"] = text_data[comments_column].apply(custom_tokenizer)
text_data.to_csv('youtube_comments_with_tokenized.csv', index=False)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(text_data["comment_text"])

n_topics = 3
lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
lsa_matrix = lsa_model.fit_transform(tfidf_matrix)
terms = tfidf_vectorizer.get_feature_names_out()

top_words_per_topic = {}

for i, comp in enumerate(lsa_model.components_):
    top_indices = comp.argsort()[-10:]
    top_terms = [terms[idx] for idx in top_indices]
    print(f"Topic {i + 1}: {', '.join(top_terms)}")
    top_words_per_topic[f"Topic {i + 1}"] = top_terms

top_words_df = pd.DataFrame.from_dict(top_words_per_topic, orient="index").T
print("\nTop Words DataFrame:")
print(top_words_df)

##Topic coherence

In [ ]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

text_data["tokenized_comments"] = text_data[comments_column].apply(word_tokenize)
text_data["filtered_comments"] = text_data["tokenized_comments"].apply(
    lambda x: [word for word in x if word not in polish_stopwords]
)
text_data["processed_comments"] = text_data["filtered_comments"].apply(" ".join)


from gensim.models import CoherenceModel
from gensim.corpora.dictionary import Dictionary

texts = text_data["filtered_comments"]
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]
n_topics = 3

from gensim.models import LsiModel
lsi_model = LsiModel(corpus, num_topics=n_topics, id2word=dictionary)

coherence_model = CoherenceModel(model=lsi_model, texts=texts, dictionary=dictionary, coherence="c_v")
coherence_score = coherence_model.get_coherence()
print(f"Coherence Score: {coherence_score}")

##Cosine Similarity for comments

In [ ]:
def filter_and_rank_comments_by_terms(comments, terms):
    scores = []
    for comment in comments:
        score = sum(1 for term in terms if term.lower() in comment.lower())
        scores.append(score)
    comment_scores = pd.DataFrame({"comment": comments, "score": scores})
    ranked_comments = comment_scores[comment_scores["score"] > 0].sort_values(by="score", ascending=False)
    return ranked_comments

ranked_comments_per_topic = {}
for topic, terms in top_words_per_topic.items():
    topic_comments = filter_and_rank_comments_by_terms(text_data["comment_text"], terms)
    ranked_comments_per_topic[topic] = topic_comments

top_comments_topic_1 = ranked_comments_per_topic["Topic 1"].head(5)
print("Top Comments for Topic 1:")
print(top_comments_topic_1)

Each comment is scored based on how many topic terms it contains.
This provides a relevance ranking for the comments.

In [ ]:
top_comments_topic_2 = ranked_comments_per_topic["Topic 2"].head(5)
print("Top Comments for Topic 2:")
print(top_comments_topic_2)

In [ ]:
top_comments_topic_3 = ranked_comments_per_topic["Topic 3"].head(5)
print("Top Comments for Topic 3:")
print(top_comments_topic_3)

##Scores in topics

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

topic_stats = []
for topic, comments in ranked_comments_per_topic.items():
    num_comments = len(comments)
    avg_score = comments['score'].mean() if num_comments > 0 else 0
    topic_stats.append({"Topic": topic, "Comments": num_comments, "Avg_Score": avg_score})

topic_stats_df = pd.DataFrame(topic_stats)

fig, ax1 = plt.subplots(figsize=(10, 6))

sns.barplot(
    x="Topic",
    y="Comments",
    data=topic_stats_df,
    ax=ax1,
    color="skyblue",
    label="Number of Comments",
)

ax2 = ax1.twinx()
sns.lineplot(
    x="Topic",
    y="Avg_Score",
    data=topic_stats_df,
    ax=ax2,
    color="orange",
    marker="o",
    label="Average Score",
)

ax1.set_title("Comments and Average Relevance Scores by Topic", fontsize=14)
ax1.set_xlabel("Topic", fontsize=12)
ax1.set_ylabel("Number of Comments", fontsize=12, color="skyblue")
ax2.set_ylabel("Average Relevance Score", fontsize=12, color="orange")
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

topic_heatmap_data = pd.DataFrame(index=ranked_comments_per_topic.keys(),
                                  columns=ranked_comments_per_topic.keys())

for topic_row, df_row in ranked_comments_per_topic.items():
    for topic_col, df_col in ranked_comments_per_topic.items():
        if topic_row == topic_col:
            avg_score = df_row['score'].mean()
        else:
            shared_indices = df_row.index.intersection(df_col.index)
            if not shared_indices.empty:
                avg_score = (df_row.loc[shared_indices, 'score'].mean() +
                             df_col.loc[shared_indices, 'score'].mean()) / 2
            else:
                avg_score = 0
        topic_heatmap_data.loc[topic_row, topic_col] = avg_score

topic_heatmap_data = topic_heatmap_data.astype(float)
plt.figure(figsize=(10, 8))
sns.heatmap(
    topic_heatmap_data,
    annot=True,
    cmap="coolwarm",
    cbar_kws={"label": "Average Relevance Score"},
    fmt=".2f",
)

plt.title("Topic Relevance Heatmap", fontsize=16)
plt.xlabel("Topic")
plt.ylabel("Topic")
plt.show()

##Hierarchical topic model (LDA)

In [ ]:
from gensim.models import LdaModel
from gensim.corpora import Dictionary

texts = text_data["filtered_comments"]

dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

n_topics = 3
lda_model = LdaModel(corpus=corpus, num_topics=n_topics, id2word=dictionary, passes=10, random_state=42)

for topic_idx in range(n_topics):
    top_words = lda_model.show_topic(topic_idx, topn=10)
    print(f"Topic {topic_idx + 1}:")
    for word, weight in top_words:
        print(f"  {word}: {weight:.4f}")

##Word embedding visualization

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
import matplotlib.pyplot as plt

texts = text_data["comment_text"].dropna().tolist()
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

try:
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
except Exception as e:
    print(f"Error during embedding: {e}")

umap_model = UMAP(
    n_neighbors=15,
    n_components=2,
    metric="cosine"
)

try:
    reduced_embeddings = umap_model.fit_transform(embeddings)
    print("UMAP dimensionality reduction complete.")
except Exception as e:
    print(f"Error during UMAP dimensionality reduction: {e}")

plt.figure(figsize=(10, 8))
plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], alpha=0.5)
plt.title("Text Embeddings Visualization (UMAP)", fontsize=16)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.show()

##CTM

In [ ]:
import tomotopy as tp
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

sample_size = 5000
text_data = text_data.sample(n=sample_size, random_state=42).reset_index(drop=True)

ctm = tp.CTModel(
    min_df=0,
    rm_top=0,
    k=3,
    smoothing_alpha=0.1,
    eta=0.01,
    seed=123
)
for doc in text_data["comment_text"]:
    ctm.add_doc(doc.split())

ctm.train(0)
for i in range(100):
    ctm.train(10)
    print(f"Iteration: {i*10+10}, LL per word: {ctm.ll_per_word}")

topic_words = {}
for k in range(ctm.k):
    print(f"\nTop 10 words of topic #{k}:")
    topic_words[k] = [word for word, _ in ctm.get_topic_words(k, top_n=10)]
    for word, prob in ctm.get_topic_words(k, top_n=10):
        print(f"{word}: {prob:.4f}")

for k, words in topic_words.items():
    probs = [prob for _, prob in ctm.get_topic_words(k, top_n=10)]
    plt.figure(figsize=(8, 5))
    plt.barh(words, probs, color='skyblue')
    plt.xlabel('Probability')
    plt.title(f'Top 10 Words in Topic {k}')
    plt.gca().invert_yaxis()
    plt.show()

Topics for random sample:
*   1st talks about the main topic of the video,
*   2nd is about creator of the first video,
*   3rd is general infromation about people involved, as well as reactors.

In [ ]:
import pandas as pd
import tomotopy as tp
from gensim.models import LdaModel
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt
from wordcloud import WordCloud

sample_size = 5000
text_data = pd.read_csv("youtube_comments_clean_stemmed.csv").sample(n=sample_size, random_state=42).reset_index(drop=True)
text_data = text_data.dropna(subset=["comment_text"])
text_data["comment_text"] = text_data["comment_text"].astype(str)
texts = text_data["comment_text"].apply(lambda x: x.split()).tolist()

dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]
n_topics = 3
lda_model = LdaModel(corpus=corpus, num_topics=n_topics, id2word=dictionary, passes=10, random_state=42)

ctm = tp.CTModel(min_df=0, rm_top=0, k=n_topics, smoothing_alpha=0.1, eta=0.01, seed=123)
for text in texts:
    ctm.add_doc(text)
ctm.train(0)
for _ in range(100):
    ctm.train(10)

lda_coherence = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
lda_coherence_score = lda_coherence.get_coherence()
print(f"LDA Coherence Score: {lda_coherence_score:.4f}")

ctm_topics = [[word for word, _ in ctm.get_topic_words(k, top_n=10)] for k in range(ctm.k)]
ctm_coherence = CoherenceModel(topics=ctm_topics, texts=texts, dictionary=dictionary, coherence='c_v')
ctm_coherence_score = ctm_coherence.get_coherence()
print(f"CTM Coherence Score: {ctm_coherence_score:.4f}")

##Sentiment

Here there is sentiment for Polish language.

Polish Bert Model

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sampled_data = pd.read_csv("youtube_comments_with_tokenized.csv")

#test on 1% of the data
sampled_data = sampled_data.sample(frac=0.01, random_state=42)
tokenizer = AutoTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
model = AutoModelForSequenceClassification.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
classifier = pipeline('zero-shot-classification', model=model, tokenizer=tokenizer)
sentiments = ["positive", "negative", "neutral"]
def classify_comment(comment):
    if pd.notna(comment):
        result = classifier(comment, candidate_labels=sentiments)
        return result['labels'][0]
    return "Unknown"
sampled_data['sentiment'] = sampled_data['tokenized_comments'].apply(classify_comment)
sentiment_counts = sampled_data['sentiment'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette="viridis")
plt.title("Sentiment Distribution in Sampled YouTube Comments")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

It's better to train it. You can use ready databases for training online.

Let's see how some machine learning models do with this dataset.

In [ ]:
from google.colab import drive
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
###I had the dataset on drive, but you can use it localy
drive.mount('/content/drive')
dataset_path = '/content/drive/My Drive/dataset'

texts, labels = [], []
for file_name in os.listdir(dataset_path):
    file_path = os.path.join(dataset_path, file_name)
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            match = re.search(r'(.*?)\s+(__label__\S+)', line)
            if match:
                text, label = match.groups()
                texts.append(text.strip())
                labels.append(label.strip())

data = pd.DataFrame({'text': texts, 'label': labels})
data['text'] = data['text'].str.lower()

X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_tfidf, y_train_encoded)
y_pred_rf = rf_classifier.predict(X_test_tfidf)

xgb_classifier = XGBClassifier(n_estimators=100, random_state=42)
xgb_classifier.fit(X_train_tfidf, y_train_encoded)
y_pred_xgb = xgb_classifier.predict(X_test_tfidf)

classifier = LogisticRegression(max_iter=200)
classifier.fit(X_train_tfidf, y_train_encoded)
y_pred_lr = classifier.predict(X_test_tfidf)

print("Random Forest Accuracy:", accuracy_score(y_test_encoded, y_pred_rf))
print("Random Forest Classification Report:")
print(classification_report(y_test_encoded, y_pred_rf))
print("XGBoost Accuracy:", accuracy_score(y_test_encoded, y_pred_xgb))
print("XGBoost Classification Report:")
print(classification_report(y_test_encoded, y_pred_xgb))
print("Logistic Regression Accuracy:", accuracy_score(y_test_encoded, y_pred_lr))
print("Logistic Regression Classification Report:")
print(classification_report(y_test_encoded, y_pred_lr))

Training

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset

from google.colab import drive
drive.mount('/content/drive')
dataset_path = '###your path to the dataset####'

texts, labels = [], []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        match = re.search(r'(.*?)\s+(__label__\S+)', line) ###this is the structure of the file I used for the training
        if match:
            text, label = match.groups()
            texts.append(text.strip())
            labels.append(label.strip())

data = pd.DataFrame({'text': texts, 'label': labels})
data['text'] = data['text'].str.lower()
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
model = AutoModelForSequenceClassification.from_pretrained("dkleczek/bert-base-polish-uncased-v1", num_labels=len(data['label'].unique()))

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
test_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

label2id = {label: idx for idx, label in enumerate(data['label'].unique())}
train_dataset = train_dataset.map(lambda x: {'label': label2id[x['label']]})
test_dataset = test_dataset.map(lambda x: {'label': label2id[x['label']]})

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)

trainer.train()
results = trainer.evaluate()
print(f"Test Results: {results}")

You can make it better by changing parameters, depending on your results.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import pandas as pd
from datasets import Dataset

texts, labels = [], []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        match = re.search(r'(.*?)\s+(__label__\S+)', line)
        if match:
            text, label = match.groups()
            texts.append(text.strip())
            labels.append(label.strip())

data = pd.DataFrame({'text': texts, 'label': labels})
data['text'] = data['text'].str.lower()
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=64)

train_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
test_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

label2id = {label: idx for idx, label in enumerate(data['label'].unique())}
train_dataset = train_dataset.map(lambda x: {'label': label2id[x['label']]})
test_dataset = test_dataset.map(lambda x: {'label': label2id[x['label']]})

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=200,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=500,
    fp16=True,
    save_steps=1000,
    dataloader_num_workers=4,
    learning_rate=5e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
)
trainer.train()

results = trainer.evaluate()
print(f"Test Results: {results}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

accuracy = accuracy_score(y_test.map(label2id), preds)
f1 = f1_score(y_test.map(label2id), preds, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

Let's apply our trained model.

In [ ]:
import pandas as pd
df = pd.read_csv('youtube_comments_with_tokenized.csv')

#random sample of 1% of the data
sampled_data = df.sample(frac=0.01, random_state=42)
print(f"Sampled data size: {sampled_data.shape}")

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import pandas as pd

model = BertForSequenceClassification.from_pretrained("###path for your trained model###")
tokenizer = BertTokenizer.from_pretrained("###path for your trained model###")

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

classifier = pipeline('zero-shot-classification', model=model, tokenizer=tokenizer)

sentiments = ["positive", "negative", "neutral"]

def classify_comment(comment):
    if pd.notna(comment):
        encoded_comment = tokenizer(comment, truncation=True, padding='max_length', max_length=538)
        result = classifier(comment, candidate_labels=sentiments, truncation=True, padding=True, max_length=512)
        return result['labels'][0]
    return "Unknown"

sampled_data['sentiment'] = sampled_data['comment_text'].apply(classify_comment)

sentiment_counts = sampled_data['sentiment'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette="viridis")
plt.title("Sentiment Distribution in Sampled YouTube Comments")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

We can also simply just use already trained model, as an example here multilingual model xlm-roberta-large-xnli was used.

In [ ]:
from transformers import pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sampled_data = pd.read_csv("youtube_comments_with_tokenized.csv")
sampled_data = sampled_data.sample(frac=0.001, random_state=42)

classifier = pipeline(
    'zero-shot-classification',
    model="joeddav/xlm-roberta-large-xnli",
    device=0
)

sentiments = ["positive", "negative", "neutral"]
def classify_comment(comment):
    if pd.notna(comment):
        result = classifier(comment, candidate_labels=sentiments, truncation=True)
        return result['labels'][0]
    return "Unknown"

sampled_data['sentiment'] = sampled_data['tokenized_comments'].apply(classify_comment)

sentiment_counts = sampled_data['sentiment'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette="viridis")
plt.title("Sentiment Distribution in Sampled YouTube Comments")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

Let's check the accuracy on the text that we used to train the previos model to compare. As it's' a zero shot model we don't need fine tuning.

In [ ]:
import torch
import pandas as pd
import re
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive

drive.mount('/content/drive')
dataset_path = '###path to your training dataset###'

texts, labels = [], []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        match = re.search(r'(.*?)\s+(__label__\S+)', line)
        if match:
            text, label = match.groups()
            texts.append(text.strip())
            labels.append(label.strip())

data = pd.DataFrame({'text': texts, 'label': labels})
data['text'] = data['text'].str.lower()

X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

tokenizer = AutoTokenizer.from_pretrained("joeddav/xlm-roberta-large-xnli")

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
test_dataset = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

label2id = {label: idx for idx, label in enumerate(data['label'].unique())}
id2label = {v: k for k, v in label2id.items()}

def convert_labels(examples):
    examples['label'] = [label2id[label] for label in examples['label']]
    return examples

train_dataset = train_dataset.map(convert_labels, batched=True)
test_dataset = test_dataset.map(convert_labels, batched=True)

num_labels = len(label2id)
model = AutoModelForSequenceClassification.from_pretrained(
    "joeddav/xlm-roberta-large-xnli", num_labels=num_labels, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

training_args = TrainingArguments(
    output_dir='###path to the model##',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=1000,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,
    fp16=True,
    load_best_model_at_end=False,
    dataloader_num_workers=2,
    learning_rate=3e-5,
    lr_scheduler_type='cosine',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": accuracy, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
results = trainer.evaluate()
print(f"\n **Final Test Results:**\n{results}")

Analysis of the sentiment within the comments

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from collections import Counter

model = BertForSequenceClassification.from_pretrained("###path to your model###")
tokenizer = BertTokenizer.from_pretrained("###path to your model###")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
df = pd.read_csv("youtube_comments_with_tokenized.csv")

sentiments = ["positive", "neutral", "negative"]

def predict_sentiment_for_words(text):
    words = text.split()
    sentiment_counts = Counter()

    for word in words:
        inputs = tokenizer(word, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        prediction = torch.argmax(logits, dim=-1).cpu().item()

        sentiment_labels = {0: "negative", 1: "neutral", 2: "positive"}
        sentiment = sentiment_labels[prediction]
        sentiment_counts[sentiment] += 1

    return sentiment_counts

sample_size = 25
random_sample_indices = random.sample(range(len(df)), sample_size)
sampled_text_data = df.iloc[random_sample_indices].copy()

sentiment_distributions = []
for comment in sampled_text_data["comment_text"]:
    sentiment_counts = predict_sentiment_for_words(comment)
    sentiment_distributions.append(sentiment_counts)

sentiment_df = pd.DataFrame(sentiment_distributions).fillna(0)
sentiment_df = sentiment_df.reindex(columns=sentiments, fill_value=0)
sentiment_df.index = sampled_text_data.index

sentiment_df.plot(kind="bar", stacked=True, color=["green", "gray", "red"], figsize=(12, 8))
plt.title("Word-Level Sentiment Distribution for Random Sample of Comments")
plt.xlabel("Comment Index (Sampled)")
plt.ylabel("Sentiment Count")
plt.xticks(rotation=45, fontsize=8)
plt.legend(["Positive", "Neutral", "Negative"], loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = BertForSequenceClassification.from_pretrained("###path to your model###")
tokenizer = BertTokenizer.from_pretrained("###path to your model###")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

text_data = pd.read_csv("youtube_comments_with_tokenized.csv")

text_data['comment_text'] = text_data['comment_text'].fillna("").astype(str)

sampled_data = text_data.sample(frac=0.01, random_state=42)

def predict_sentiment(texts, batch_size=16):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()
        all_probs.extend(probabilities)

    return np.array(all_probs)

sentiment_probs = predict_sentiment(sampled_data["comment_text"].tolist())
sentiment_scores = sentiment_probs[:, 2] - sentiment_probs[:, 0]

sampled_data["average_sentiment"] = sentiment_scores

plt.figure(figsize=(10, 6))
sns.histplot(sampled_data["average_sentiment"], bins=50, kde=True, color="blue")
plt.title("Distribution of Average Sentiment for Sampled Comments")
plt.xlabel("Average Sentiment Score")
plt.ylabel("Frequency")
plt.show()
sampled_data[["comment_text", "average_sentiment"]].head()

Analysis of the multilingual model

In [ ]:
import pandas as pd
import random
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

labels = ["positive", "negative", "neutral"]

sample_size = 10
random_sample_indices = random.sample(range(len(text_data)), sample_size)
sampled_text_data = text_data.iloc[random_sample_indices]

sampled_results = []
for comment in sampled_text_data["processed_comments"]:
    result = classifier(comment, candidate_labels=labels)
    sampled_results.append({
        "comment": comment,
        "classification": result["labels"][0],
        "scores": dict(zip(result["labels"], result["scores"]))
    })

results_df = pd.DataFrame(sampled_results)

print("Zero-Shot Classification Results for Random Sample:")
print(results_df[["comment", "classification"]])

results_df.to_csv("zero_shot_classification_results.csv", index=False)

In [ ]:
import ast

def parse_scores(score):
    try:
        if isinstance(score, str):
            return ast.literal_eval(score)
        return score
    except Exception:
        return {}

results_df["scores"] = results_df["scores"].apply(parse_scores)

scores_df = pd.DataFrame(results_df["scores"].tolist())

average_scores = scores_df.mean()

plt.figure(figsize=(8, 6))
average_scores.plot(kind="bar", color=["green", "red", "gray"])
plt.title("Average Confidence Scores by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Average Confidence Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.show()

classification_counts = results_df["classification"].value_counts()
plt.figure(figsize=(8, 6))
classification_counts.plot(kind="bar", color=["green", "red", "gray"])
plt.title("Zero-Shot Classification: Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Comments")
plt.xticks(rotation=0)
plt.show()

print(results_df.head())

##Bag of words

In [ ]:
import pandas as pd
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import LdaModel
import tomotopy as tp
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import ast

text_data = pd.read_csv("youtube_comments_with_tokenized.csv")
text_data.dropna(subset=["comment_text"], inplace=True)
text_data = text_data[text_data["comment_text"] != ""].reset_index(drop=True)
sampled_data = text_data.sample(frac=0.01, random_state=42)

polish_model = AutoModelForSequenceClassification.from_pretrained("###path to your trained model###")
polish_tokenizer = AutoTokenizer.from_pretrained("###path to your trained model###")
polish_sentiment_analyzer = pipeline("sentiment-analysis", model=polish_model, tokenizer=polish_tokenizer)

zero_shot_classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")
sentiments = ["positive", "negative", "neutral"]
###map your labels, these are from my training dataset###
sentiment_label_mapping = {
    "__label__z_minus_m": "negative",
    "__label__z_zero": "neutral",
    "__label__z_plus_m": "positive",
    "__label__z_amb": "unknown"
}

def get_polish_sentiment(text):
    if text:
        sentiment = polish_sentiment_analyzer(text)
        return sentiment[0]['label']
    return "neutral"

sampled_data["polish_sentiment"] = sampled_data["comment_text"].apply(get_polish_sentiment)
sampled_data["polish_sentiment"] = sampled_data["polish_sentiment"].map(sentiment_label_mapping)

def get_zero_shot_sentiment(text):
    if text:
        result = zero_shot_classifier(text, candidate_labels=sentiments, truncation=True)
        return result['labels'][0]
    return "neutral"

sampled_data["zero_shot_sentiment"] = sampled_data["comment_text"].apply(get_zero_shot_sentiment)

sampled_data["tokenized_comments"] = sampled_data["tokenized_comments"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

dictionary = Dictionary(sampled_data["tokenized_comments"])
gensim_bow = [dictionary.doc2bow(text) for text in sampled_data["tokenized_comments"]]
lda_model = LdaModel(corpus=gensim_bow, num_topics=3, id2word=dictionary, passes=10, random_state=42)

print("\nLDA Topics:")
for topic_idx, topic in lda_model.print_topics(num_words=5):
    print(f"Topic {topic_idx + 1}: {topic}")

ctm = tp.CTModel(min_df=0, rm_top=0, k=3, smoothing_alpha=0.1, eta=0.01, seed=123)
for tokens in sampled_data["tokenized_comments"]:
    ctm.add_doc(tokens)

ctm.train(0)
for _ in range(50):
    ctm.train(10)

print("\nCTM Topics:")
for k in range(ctm.k):
    topic_words = [word for word, _ in ctm.get_topic_words(k, top_n=5)]
    print(f"Topic {k + 1}: {topic_words}")

print("\nSentiment Distribution using Polish Model:")
print(sampled_data["polish_sentiment"].value_counts())
print("\nSentiment Distribution using Zero-Shot Model:")
print(sampled_data["zero_shot_sentiment"].value_counts())

plt.figure(figsize=(10, 6))
sns.countplot(data=sampled_data, x='polish_sentiment', palette="Set2")
plt.title("Sentiment Distribution (Polish BERT Model)")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 6))
sns.countplot(data=sampled_data, x='zero_shot_sentiment', palette="Set2")
plt.title("Sentiment Distribution (Zero-Shot Classification)")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

sampled_data[["comment_text", "polish_sentiment", "zero_shot_sentiment"]]

##Hate speech



Let's look at negative comments (according to the model)

In [ ]:
negative_comments = sampled_data[sampled_data["polish_sentiment"] == "negative"]['comment_text']

negative_comments_cleaned = negative_comments.apply(clean_text)

print(negative_comments_cleaned.head())

Top negative words

In [ ]:
from collections import Counter

negative_tokens = [token for comment in negative_comments_cleaned for token in comment.split()]

negative_word_freq = Counter(negative_tokens)

top_10_negative_words = negative_word_freq.most_common(10)
print("Top 10 Words in Negative Comments:")
print(top_10_negative_words)

Bigrams and trigrams

In [ ]:
from collections import Counter

negative_comments = sampled_data[sampled_data["polish_sentiment"] == "negative"]['comment_text']
negative_comments_cleaned = negative_comments.apply(clean_text)
negative_tokens = [token for comment in negative_comments_cleaned for token in comment.split()]
bigrams = [tuple(negative_tokens[i:i+2]) for i in range(len(negative_tokens)-1)]
trigrams = [tuple(negative_tokens[i:i+3]) for i in range(len(negative_tokens)-2)]
bigram_freq = Counter(bigrams)
trigram_freq = Counter(trigrams)
top_10_bigrams = bigram_freq.most_common(10)
top_10_trigrams = trigram_freq.most_common(10)

print("Top 10 Bigrams in Negative Comments:")
print(top_10_bigrams)
print("\nTop 10 Trigrams in Negative Comments:")
print(top_10_trigrams)

Trend of negative words

In [ ]:
import matplotlib.pyplot as plt

sampled_data['comment_date'] = pd.to_datetime(sampled_data['comment_date'])
negative_comments_with_dates = sampled_data[sampled_data["polish_sentiment"] == "negative"]
negative_comments_with_dates['date_hour'] = negative_comments_with_dates['comment_date'].dt.strftime('%Y-%m-%d %H')
negative_comments_by_time = negative_comments_with_dates.groupby('date_hour').size()

plt.figure(figsize=(12, 6))
negative_comments_by_time.plot(kind='line', title="Trend of Negative Comments by Day and Hour")
plt.xlabel('Date and Hour')
plt.ylabel('Number of Negative Comments')
plt.xticks(rotation=90)
plt.show()

Co-occuring negative words

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

G = nx.Graph()

for comment in negative_comments_cleaned:
    tokens = comment.split()
    for i in range(len(tokens)-1):
        word1, word2 = tokens[i], tokens[i+1]
        if word1 != word2:
            if G.has_edge(word1, word2):
                G[word1][word2]['weight'] += 1
            else:
                G.add_edge(word1, word2, weight=1)

word_freq = Counter([token for comment in negative_comments_cleaned for token in comment.split()])
top_words = [word for word, _ in word_freq.most_common(50)]  #top 50 as there is too much words and nothing is visible

top_words_network = G.subgraph(top_words)
node_sizes = [word_freq[node] * 10 for node in top_words_network.nodes]
edge_weights = [G[u][v]['weight'] for u, v in top_words_network.edges]

plt.figure(figsize=(12, 10))
nx.draw_networkx(top_words_network,
                 node_size=node_sizes,
                 width=edge_weights,
                 font_size=12,
                 alpha=0.7,
                 with_labels=True,
                 node_color='lightblue',
                 font_weight='bold',
                 edge_color=edge_weights,
                 edge_cmap=plt.cm.Blues)
plt.title("Negative Word Co-occurrence Network (Top 50 Words)")
plt.axis('off')
plt.show()

##Vader Language model

You can use this one for english language

In [ ]:
!pip install vaderSentiment

In [ ]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
from langdetect import detect
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
import matplotlib.pyplot as plt
text_data = pd.read_csv("youtube_comments_clean_stemmed.csv")
text_data.dropna(subset=["comment_text"], inplace=True)
text_data = text_data[text_data["comment_text"] != ""]

In [ ]:
text_data["tokenized_comments"] = text_data["comment_text"].apply(word_tokenize)
max_tokens = 512
text_data["processed_comments"] = text_data["tokenized_comments"].apply(" ".join)
def analyze_and_save(comments, output_file, batch_size=32, start_index=0):
    sentiments = []
    for i in tqdm(range(start_index, len(comments), batch_size), desc="Processing Batches", ncols=100):
        batch = comments[i:i+batch_size]

        batch = [comment[:max_tokens] for comment in batch]  # Truncate to max_tokens length

        try:
            results = sentiment_analyzer(batch)
            sentiments_batch = [result["label"] for result in results]
        except Exception as e:
            print(f"Error processing batch starting at index {i}: {e}")
            sentiments_batch = ["Error"] * len(batch)

        sentiments.extend(sentiments_batch)

        temp_data = text_data.iloc[:i+len(batch)].copy()
        temp_data["sentiment"] = sentiments
        temp_data.to_csv(output_file, index=False)

    return sentiments

In [ ]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
import pandas as pd
nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

In [ ]:
# Function to classify sentiment of a word
def classify_word_sentiment(word):
    score = sia.polarity_scores(word)['compound']
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'
text_data['word_sentiments'] = text_data['tokenized_comments'].apply(
    lambda tokens: [classify_word_sentiment(word) for word in tokens]
)
def sentiment_count(sentiment_list):
    return {
        'positive': sentiment_list.count('positive'),
        'neutral': sentiment_list.count('neutral'),
        'negative': sentiment_list.count('negative')
    }

text_data['sentiment_counts'] = text_data['word_sentiments'].apply(sentiment_count)
total_sentiments = text_data['sentiment_counts'].apply(pd.Series).sum()
print(total_sentiments)

In [ ]:
import matplotlib.pyplot as plt
sentiment_totals = text_data['sentiment_counts'].apply(pd.Series).sum()
sentiment_totals.plot(kind='bar', color=['green', 'gray', 'red'])
plt.title("Sentiment Distribution (Positive, Neutral, Negative)")
plt.ylabel("Count")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import random

sample_size = 25
random_sample_indices = random.sample(range(len(text_data)), sample_size)
sampled_text_data = text_data.iloc[random_sample_indices]

sampled_sentiment_counts_df = sampled_text_data['sentiment_counts'].apply(pd.Series)
sampled_sentiment_counts_df['comment_index'] = sampled_text_data.index

sampled_sentiment_counts_df.set_index('comment_index').plot(
    kind='bar',
    stacked=True,
    color=['green', 'gray', 'red'],
    figsize=(12, 8)
)
plt.title("Sentiment Distribution for Random Sample of Comments (Based on Words)")
plt.xlabel("Comment Index (Sampled)")
plt.ylabel("Sentiment Count")
plt.xticks(rotation=45, fontsize=8)
plt.legend(['Positive', 'Neutral', 'Negative'], loc='upper right')
plt.tight_layout()
plt.show()

##Sources


* dkleczek/bert-base-polish-uncased-v1, Hugging Face<br> https://huggingface.co/dkleczek/bert-base-polish-uncased-v1 <br>
* joeddav/xlm-roberta-large-xnli, Hugging Face<br> https://huggingface.co/joeddav/xlm-roberta-large-xnli <br>

* YouTube Data API Overview, Google for Developers <br>https://developers.google.com/youtube/v3/getting-started

